In [0]:
from pyspark.sql import functions as F
stg_df = spark.read.table("gh_archive_lakehouse_dev.default.stg_gh_archive")

In [0]:
def daily_event_count_by_type(stg_df):
    return (
        stg_df
        .groupBy("event_date", "type")
        .agg(F.count("*").alias("event_count"))
        .orderBy("event_date", "type")
    )

daily_event_count_by_type = daily_event_count_by_type(stg_df)
daily_event_count_by_type.write.mode("overwrite").saveAsTable("mart_daily_event_count_by_type")

Databricks visualization. Run in Databricks to view.

In [0]:
def top_repos(stg_df):
    return (
        stg_df
        .groupBy("repo_name")
        .agg(F.count("*").alias("count"))
        .orderBy(F.desc("count"))
        .limit(20)
    )

top_repos = top_repos(stg_df)
top_repos.write.mode("overwrite").saveAsTable("mart_top_repos")

In [0]:
def pr_actions(stg_df):
    return (
        stg_df
        .filter(F.col("type") == "PullRequestEvent")
        .groupBy(F.variant_get(F.col("payload"),"$.action","string").alias("action"))
        .agg(F.count("*").alias("count"))
        .orderBy(F.desc("count"))
    )

pr_actions = pr_actions(stg_df)
pr_actions.write.mode("overwrite").saveAsTable("mart_pr_actions")


In [0]:
def human_bot_count(stg_df):
    return (
        stg_df
        .groupBy(
            F.when(F.col("actor_login").endswith("[bot]"), "bot")
            .otherwise("human")
            .alias("user_type")
        )
        .agg(F.count("*").alias("count"))
    )

human_bot_count = human_bot_count(stg_df)
human_bot_count.write.mode("overwrite").saveAsTable("mart_human_bot_count")